# Data Exploration of different datasets

## Datasets used in this project

- [MARS dataset](https://www.sciencedirect.com/science/article/pii/S2352340923000604)
- [ITM-Rec dataset](https://arxiv.org/abs/2303.10230)

In [37]:
from typing import Any, Self

import numpy as np
import pandas as pd
import tiktoken
from sklearn import set_config
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import (
    FunctionTransformer,
    LabelEncoder,
    MinMaxScaler,
    OrdinalEncoder,
    MultiLabelBinarizer,
)

## Configuration

In [38]:
ITEM_COL: str = "item_id"
USER_COL: str = "user_id"
TIME_COL: str = "timestamp"
RATING_COL: str = "rating"
RELEVANT_COL: str = "relevant"
DATA_FOLDER: str = "../data"
TEST_SIZE: float = 0.4
VAL_SIZE: float = 0.2
RANDOM_STATE: int = 42
PADDING_VAL: int = 0
INIT_TOKEN: str = "<|init|>"
END_TOKEN: str = "<|end|>"
EMPTY_TOKEN: str = "<|empty|>"

## Data Processor

In [39]:
set_config(transform_output="pandas")


def clean_and_process_df(df: pd.DataFrame) -> None:
    if RELEVANT_COL not in df.columns:
        # An item is relevant if its rating is greater or equal than the threshold
        # The threshold is the mean of the ratings of the user
        mean_user_ratings = df.groupby(USER_COL)[RATING_COL].transform("mean")
        df[RELEVANT_COL] = df[RATING_COL] >= mean_user_ratings

    df.columns = (
        df.columns.str.lower()
        .str.strip()
        .str.replace(" ", "_")
        .str.replace(r"[^\w]", "", regex=True)  # Quita símbolos como ?, !, @
    )


def get_column_types(df, id_cols):
    exclude_cols = [RATING_COL, RELEVANT_COL, TIME_COL]
    numeric_cols, categorical_cols, list_cols, text_cols = [], [], [], []

    for col in df.columns:
        if col in exclude_cols:
            continue

        if col in id_cols:
            categorical_cols.append(col)
            continue

        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_cols.append(col)
            continue

        non_null_series = df[col].dropna()
        if non_null_series.empty:
            continue
        sample_value = non_null_series.iloc[0]

        if isinstance(sample_value, list) or (
            isinstance(sample_value, str) and sample_value.startswith("[")
        ):
            list_cols.append(col)
        elif isinstance(sample_value, str):
            num_unique = df[col].nunique()
            avg_words = non_null_series.str.split().str.len().mean()
            # Si tiene muchas palabras o es casi único, es texto
            if avg_words > 4 or (num_unique / len(df)) > 0.5:
                text_cols.append(col)
            else:
                categorical_cols.append(col)
        else:
            categorical_cols.append(col)

    return numeric_cols, categorical_cols, list_cols, text_cols


class TokenizerTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, model_name="cl100k_base", max_length=128):
        self.model_name = model_name
        self.max_length = max_length
        self._enc = tiktoken.get_encoding(self.model_name)

    def fit(self, X, y=None) -> Self:
        return self

    def transform(self, X: pd.DataFrame | np.ndarray) -> pd.DataFrame:
        X_df = pd.DataFrame(X)

        def tokenize_and_pad(text: str):
            text = f"{INIT_TOKEN} {text} {END_TOKEN}"
            tokens = self._enc.encode(
                text, allowed_special={INIT_TOKEN, END_TOKEN, EMPTY_TOKEN}
            )
            tokens = tokens[: self.max_length]
            return tokens + [PADDING_VAL] * (self.max_length - len(tokens))

        res = X_df.iloc[:, 0].fillna(EMPTY_TOKEN).apply(tokenize_and_pad)

        # Devolvemos un DataFrame donde cada fila es la lista de tokens
        # IMPORTANTE: Para que el pipeline de SKLearn no explote,
        # a veces es mejor devolverlo como una serie de objetos.
        return pd.DataFrame({X_df.columns[0]: res})


class TimeFeaturesTransformer(BaseEstimator, TransformerMixin):
    """
    Entrada: una columna TIME_COL con strings/datetime.
    Salida: DataFrame con features numéricas (float) listas para imputar/escalar.
    """

    def fit(self, X, y=None) -> Self:
        return self

    def transform(self, X: pd.DataFrame | np.ndarray) -> pd.DataFrame:
        X_df = pd.DataFrame(X)
        s = X_df.iloc[:, 0]

        dt = pd.to_datetime(s, errors="coerce", utc=True)

        # timestamp en segundos (float para permitir NaN)
        ts = (dt.astype(np.int64) // 10**9).astype("float64")

        hour = dt.dt.hour.astype("float64")
        dow = dt.dt.dayofweek.astype("float64")  # 0=lunes
        month = dt.dt.month.astype("float64")

        # cíclicas (mejor para redes que quieren continuidad)
        hour_sin = np.sin(2 * np.pi * (hour / 24.0))
        hour_cos = np.cos(2 * np.pi * (hour / 24.0))
        dow_sin = np.sin(2 * np.pi * (dow / 7.0))
        dow_cos = np.cos(2 * np.pi * (dow / 7.0))

        return pd.DataFrame(
            {
                "time_ts": ts,
                "time_hour": hour,
                "time_dow": dow,
                "time_month": month,
                "time_hour_sin": hour_sin,
                "time_hour_cos": hour_cos,
                "time_dow_sin": dow_sin,
                "time_dow_cos": dow_cos,
            },
            index=X_df.index,
        )


class DataProcessor:
    def __init__(
        self,
        id_cols: list[str],
        numeric_cols: list[str],
        categorical_cols: list[str],
        text_cols: list[str],
        list_cols: list[str],
        tokenizer: str = "cl100k_base",
        max_length: int = 128,
    ):
        self.id_cols = id_cols
        self.categorical_cols = categorical_cols
        self.numeric_cols = numeric_cols
        self.text_cols = text_cols
        self.list_cols = list_cols
        self.tokenizer = tokenizer
        self.max_length = max_length

        self.pipeline = self._build_pipeline()

    def _build_pipeline(self) -> Pipeline:
        transformers = []

        transformers.append(
            (
                "time",
                Pipeline(
                    [
                        ("time_feats", TimeFeaturesTransformer()),
                        ("imputer", SimpleImputer(strategy="median")),
                        ("scaler", MinMaxScaler()),
                        (
                            "to_f32",
                            FunctionTransformer(
                                lambda x: x.astype(np.float32),
                                feature_names_out="one-to-one",
                            ),
                        ),
                    ]
                ),
                [TIME_COL],
            )
        )

        if self.numeric_cols:
            transformers.append(
                (
                    "num",
                    Pipeline(
                        [
                            ("imputer", SimpleImputer(strategy="mean")),
                            ("scaler", MinMaxScaler()),
                            (
                                "to_f32",
                                FunctionTransformer(
                                    lambda x: x.astype(np.float32),
                                    feature_names_out="one-to-one",
                                ),
                            ),
                        ]
                    ),
                    self.numeric_cols,
                )
            )

        if self.categorical_cols:
            transformers.append(
                (
                    "cat",
                    Pipeline(
                        [
                            (
                                "imputer",
                                SimpleImputer(
                                    strategy="constant", fill_value="Undefined"
                                ),
                            ),
                            (
                                "encoder",
                                OrdinalEncoder(
                                    handle_unknown="use_encoded_value",
                                    unknown_value=-1,
                                ),
                            ),
                            # Shift to 1-based indexing
                            (
                                "shift_plus1",
                                FunctionTransformer(
                                    lambda x: (x + 1).astype(np.int64),
                                    feature_names_out="one-to-one",
                                ),
                            ),
                        ]
                    ),
                    self.categorical_cols,
                )
            )

        if self.text_cols:
            for col in self.text_cols:
                transformers.append(
                    (
                        f"text_{col}",
                        TokenizerTransformer(self.tokenizer, self.max_length),
                        [col],
                    )
                )

        transformers.append(("rating_raw", "passthrough", [RATING_COL]))
        transformers.append(("relevant_raw", "passthrough", [RELEVANT_COL]))

        ct = ColumnTransformer(transformers, remainder="drop", sparse_threshold=0)
        # ct.set_output(transform="pandas")

        return Pipeline(steps=[("preprocessor", ct)])

    def fit_transform(
        self,
        train_df: pd.DataFrame,
        val_df: pd.DataFrame,
        test_df: pd.DataFrame | None,
    ) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame | None]:
        feats = (
            self.numeric_cols
            + self.categorical_cols
            + self.text_cols
            + [TIME_COL, RATING_COL, RELEVANT_COL]
        )

        train_processed = self.pipeline.fit_transform(train_df[feats])
        val_processed = self.pipeline.transform(val_df[feats])

        test_processed = None
        if test_df is not None:
            test_processed = self.pipeline.transform(test_df[feats])

        assert isinstance(train_processed, pd.DataFrame)
        assert isinstance(val_processed, pd.DataFrame)

        if test_processed is not None:
            assert isinstance(test_processed, pd.DataFrame)

        for df_p in [train_processed, val_processed, test_processed]:
            if df_p is not None:
                df_p.columns = [c.split("__")[-1] for c in df_p.columns]

        return train_processed, val_processed, test_processed

## MARS Dataset

In [40]:
def load_mars() -> pd.DataFrame:
    """Load and preprocess the MARS dataset.

    This loader combines English and French rating files and merges them with
    item metadata. It also standardizes column names and ensures consistent
    schema across sources.

    Returns:
        pd.DataFrame: The MARS dataset.
    """
    explicit_df_en = pd.read_csv(f"{DATA_FOLDER}/raw/mars/explicit_ratings_en.csv")
    explicit_df_fr = pd.read_csv(f"{DATA_FOLDER}/raw/mars/explicit_ratings_fr.csv")

    items_en = pd.read_csv(f"{DATA_FOLDER}/raw/mars/items_en.csv")
    items_fr = pd.read_csv(f"{DATA_FOLDER}/raw/mars/items_fr.csv")

    df_explicit = pd.concat([explicit_df_en, explicit_df_fr], ignore_index=True)
    df_items = pd.concat([items_en, items_fr], ignore_index=True)

    df_items = df_items.drop(columns=["created_at"])

    df = pd.merge(df_explicit, df_items, on=ITEM_COL, how="inner")

    df.rename(
        columns={
            "user_id": USER_COL,
            "item_id": ITEM_COL,
            "rating": RATING_COL,
            "Difficulty": "difficulty",
            "type": "item_type",
            "created_at": TIME_COL,
        },
        inplace=True,
    )

    df = df.drop(columns=["Job", "Software", "Theme"])

    clean_and_process_df(df)

    return df


df = load_mars()
df

,user_id,item_id,watch_percentage,timestamp,rating,language,name,nb_views,description,difficulty,duration,item_type,relevant
0,224557,510,100,2018-09-28 16:18:29,10,en,What is OneDrive for Business?,1114.0,OneDrive for Businessis an online libraryto st...,Beginner,42.0,tutorial,True
1,224557,615,100,2018-09-28 16:22:22,10,en,Tell me what you want to do,184.0,Tell me brings featuresand helps topic to your...,Beginner,57.0,tutorial,True
2,224557,7680,100,2018-09-28 16:23:34,10,en,Create a meeting in the group calendar,73.0,The Groups calendar helps youto track all the ...,Intermediate,72.0,tutorial,True
3,224293,510,100,2018-09-28 17:20:30,10,en,What is OneDrive for Business?,1114.0,OneDrive for Businessis an online libraryto st...,Beginner,42.0,tutorial,True
4,224293,515,100,2018-09-28 17:40:02,10,en,Work with documents in a synced library folder...,253.0,Once you sync your one drive library to your c...,Beginner,87.0,tutorial,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...
88993,610452,419834,92,2021-09-21 15:14:13,10,fr,Présentation générale de Shift,42.0,NaN,NaN,83.0,tutorial,True
88994,610452,419835,100,2021-09-21 15:15:40,10,fr,Présentation de l'interface Shift,37.0,NaN,NaN,66.0,tutorial,True
88995,610452,419839,99,2021-09-21 15:16:58,10,fr,Qu'est-ce qu'un shift ouvert ?,33.0,NaN,NaN,40.0,tutorial,True
88996,610452,419841,100,2021-09-21 15:17:54,10,fr,Compléter le planning et le partager,37.0,NaN,NaN,99.0,tutorial,True


## Exploration

In [41]:
df[RATING_COL].value_counts()

rating
10    70336
1      3317
9      3003
2      2424
8      2006
3      1826
4      1639
6      1507
5      1478
7      1462
Name: count, dtype: int64

## Splitting

In [42]:
df = df.sort_values(by=TIME_COL)
train_val_df, test_df = train_test_split(
    df,
    test_size=TEST_SIZE,
    shuffle=False,
    random_state=RANDOM_STATE,
)
train_df, val_df = train_test_split(
    train_val_df,
    test_size=VAL_SIZE,
    shuffle=False,
    random_state=RANDOM_STATE,
)

train_df[RATING_COL].value_counts()

rating
10    33224
1      1807
2      1501
9      1379
8       976
3       945
4       841
5       711
6       681
7       653
Name: count, dtype: int64

## Preprocessing

In [43]:
id_cols = [USER_COL, ITEM_COL]

num_cols, cat_lengths, list_cols, text_cols = get_column_types(df, id_cols)

preprocessor = DataProcessor(
    numeric_cols=num_cols,
    categorical_cols=cat_lengths,
    text_cols=text_cols,
    list_cols=list_cols,
    id_cols=id_cols,
)

train_df, val_df, test_df = preprocessor.fit_transform(train_df, val_df, test_df)

train_df

,time_ts,time_hour,time_dow,time_month,time_hour_sin,time_hour_cos,time_dow_sin,time_dow_cos,watch_percentage,nb_views,duration,user_id,item_id,language,difficulty,item_type,name,description,rating,relevant
2911,0.000000e+00,0.608696,0.666667,0.727273,0.25000,0.066987,0.277479,0.0,0.01,0.021303,0.045282,940,793,1,4,1,"[27, 91, 2381, 91, 29, 3331, 29438, 83739, 408...","[27, 91, 2381, 91, 29, 7357, 1938, 2626, 7640,...",1,False
2912,1.837276e-07,0.608696,0.666667,0.727273,0.25000,0.066987,0.277479,0.0,0.06,0.012369,0.049080,940,795,1,4,1,"[27, 91, 2381, 91, 29, 29438, 311, 7572, 48153...","[27, 91, 2381, 91, 29, 5884, 990, 449, 279, 75...",1,False
2913,4.409462e-06,0.608696,0.666667,0.727273,0.25000,0.066987,0.277479,0.0,1.00,0.008384,0.037102,940,796,1,4,1,"[27, 91, 2381, 91, 29, 32406, 701, 423, 77749,...","[27, 91, 2381, 91, 29, 4740, 1521, 2478, 62469...",10,True
3206,6.724430e-06,0.608696,0.666667,0.727273,0.25000,0.066987,0.277479,0.0,1.00,0.062672,0.023663,1105,882,1,4,1,"[27, 91, 2381, 91, 29, 29438, 311, 40713, 8373...","[27, 91, 2381, 91, 29, 17375, 76254, 84633, 37...",10,True
3207,1.289768e-05,0.608696,0.666667,0.727273,0.25000,0.066987,0.277479,0.0,1.00,0.015256,0.027461,1105,883,1,4,1,"[27, 91, 2381, 91, 29, 51968, 315, 279, 17963,...","[27, 91, 2381, 91, 29, 763, 420, 2835, 11, 584...",10,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
28120,9.999613e-01,0.565217,0.666667,0.454545,0.37059,0.017037,0.277479,0.0,1.00,0.100192,0.017821,3762,1528,2,4,1,"[27, 91, 2381, 91, 29, 2394, 5512, 306, 367, 4...","[27, 91, 2381, 91, 29, 14094, 778, 23577, 2107...",10,True
60566,9.999909e-01,0.565217,0.666667,0.454545,0.37059,0.017037,0.277479,0.0,1.00,0.186229,0.039147,897,1571,2,4,1,"[27, 91, 2381, 91, 29, 12535, 1760, 978, 1414,...","[27, 91, 2381, 91, 29, 27058, 9189, 91878, 386...",10,True
61508,9.999971e-01,0.565217,0.666667,0.454545,0.37059,0.017037,0.277479,0.0,1.00,0.048378,0.036810,64,1613,2,4,1,"[27, 91, 2381, 91, 29, 32567, 67954, 90330, 25...","[27, 91, 2381, 91, 29, 13789, 30362, 4864, 367...",10,True
4562,9.999974e-01,0.565217,0.666667,0.454545,0.37059,0.017037,0.277479,0.0,0.16,0.301952,0.024540,4292,1536,2,4,1,"[27, 91, 2381, 91, 29, 2394, 5512, 306, 367, 4...","[27, 91, 2381, 91, 29, 3489, 438, 3900, 274, 6...",2,False
